# PesaPap Growth Analytics: SQL, KPIs and an Onboarding Experiment

**Audience:** Head of Growth. **Data:** 3,000 users, 12,000 transactions (15 Jan to 13 Jun 2026), one onboarding A/B test.

Sections: 1 Database build, 2 SQL (validation and a-h), 3 KPI scorecard, 4 RFM, 5 Cohort retention, 6 Experiment, 7 Business impact.

## 1. Build `pesapap.db` from the CSVs

The CSVs are produced by `generate_data.py` (seed 77). Keys: `users.user_id` PK; `transactions.txn_id` PK, `transactions.user_id` FK to users; `experiment.user_id` PK and FK to users.

In [1]:
import os, re, sqlite3, subprocess, sys
import numpy as np, pandas as pd
from scipy import stats
pd.set_option("display.width", 160); pd.set_option("display.max_columns", 30)

if not os.path.exists("users.csv"):
    subprocess.run([sys.executable, "generate_data.py"], check=True)

if os.path.exists("pesapap.db"): os.remove("pesapap.db")
con = sqlite3.connect("pesapap.db")
con.execute("PRAGMA foreign_keys = ON")
con.executescript('''
CREATE TABLE users (
    user_id INTEGER PRIMARY KEY,
    signup_date TEXT NOT NULL,
    county TEXT NOT NULL,
    acquisition_channel TEXT NOT NULL
);
CREATE TABLE transactions (
    txn_id INTEGER PRIMARY KEY,
    user_id INTEGER NOT NULL,
    txn_date TEXT NOT NULL,
    product TEXT NOT NULL,
    amount REAL NOT NULL,
    FOREIGN KEY (user_id) REFERENCES users(user_id)
);
CREATE TABLE experiment (
    user_id INTEGER PRIMARY KEY,
    "group" TEXT NOT NULL CHECK ("group" IN ('A','B')),
    converted INTEGER NOT NULL CHECK (converted IN (0,1)),
    FOREIGN KEY (user_id) REFERENCES users(user_id)
);
''')
for t in ["users", "transactions", "experiment"]:
    pd.read_csv(f"{t}.csv").to_sql(t, con, if_exists="append", index=False)
con.commit()
print(pd.read_sql("PRAGMA foreign_key_check", con).shape[0], "foreign-key violations")
for t in ["users", "transactions", "experiment"]:
    print(t, pd.read_sql(f"SELECT COUNT(*) n FROM {t}", con).n[0], "rows")

0 foreign-key violations
users 3000 rows
transactions 12000 rows
experiment 3000 rows


## 2. SQL queries

Every query lives in `sql/queries.sql`, each preceded by a comment with its business question. The cell below splits the file and runs each query through `pd.read_sql`.

In [2]:
sql_text = open("sql/queries.sql").read()
parts = re.split(r"-- ===== (\w+) =====", sql_text)[1:]
Q = dict(zip(parts[::2], parts[1::2]))
def run(name):
    q = Q[name]
    question = re.search(r"-- Business question[^\n]*(?:\n--[^\n]*)*", q).group(0)
    print(question.replace("-- ", "").strip(), "\n")
    return pd.read_sql(q, con)
print(list(Q))

['validation', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h']


### 2.0 Data validation: transactions before the user's signup date

In [3]:
val = run('validation'); val

Business question: How many transactions happen BEFORE the user's signup
date (impossible in a real product), and what share of the ledger and of
value do they represent? 



,txns_before_signup,pct_of_all_txns,value_before_signup_kes,users_affected
0,3737,31.14,5022531.0,1749


**What this implies.** About 31% of transactions (about KES 5.0M of KES 16.0M) are dated before the user signed up. A user cannot transact before having an account, so this is a data-integrity problem: signup dates and transaction dates were captured or generated independently, or signup dates were overwritten/back-filled. In this synthetic dataset the cause is simply that the two date columns are drawn independently. In a real company I would raise it with data engineering.

**How I treat them.**
- *Headline money totals (SQL a-h, KPI scorecard):* kept, because the dataset is the only ledger we have and dropping 31% would change every number. I also show a "clean" version for comparison.
- *User-lifecycle analysis (RFM and cohort retention):* excluded, because "months since signup" is negative for these rows and cannot be interpreted. Behaviour before signup cannot be attributed to the user's journey.
- The results are reported with this limitation; nothing is silently deleted.

### 2a. Total transaction value and count by product

In [4]:
run('a')

Business question (a): What is the total transaction value and the number
of transactions for each product? 



,product,txn_count,total_value_kes
0,Send Money,4161,5639308.0
1,Airtime,3662,4708104.0
2,Bill Pay,2388,3224392.0
3,Merchant Pay,1789,2407505.0


### 2b. Top five counties by value

In [5]:
run('b')

Business question (b): Which five counties generate the most transaction
value? 



,county,total_value_kes,txn_count
0,Mombasa,3464271.0,2616
1,Nairobi,3403364.0,2549
2,Nakuru,3138079.0,2338
3,Kisumu,3059055.0,2321
4,Eldoret,2914540.0,2176


### 2c. Monthly value and running total

In [6]:
run('c')

Business question (c): How does monthly transaction value evolve, and what
is the cumulative (running) value over time? 



,month,monthly_value_kes,running_total_kes
0,2026-01,1733121.0,1733121.0
1,2026-02,2993105.0,4726226.0
2,2026-03,3199125.0,7925351.0
3,2026-04,3262447.0,11187798.0
4,2026-05,3365002.0,14552800.0
5,2026-06,1426509.0,15979309.0


### 2d. Value by acquisition channel (JOIN)

In [7]:
run('d')

Business question (d): Which acquisition channels bring the most
transaction value (and how much per transaction)? 



,acquisition_channel,txn_count,total_value_kes,avg_txn_value_kes
0,Search,3192,4279089.0,1341.0
1,Agent,2919,3954250.0,1355.0
2,Social,3008,3926296.0,1305.0
3,Referral,2881,3819674.0,1326.0


### 2e. Top ten users by value (RANK)

In [8]:
run('e')

Business question (e): Who are the ten highest-value users, ranked by total
transaction value? 



,user_id,total_value_kes,value_rank
0,1962,29244.0,1
1,2643,29144.0,2
2,2894,28008.0,3
3,2671,27317.0,4
4,50,27158.0,5
5,2279,26634.0,6
6,2837,25643.0,7
7,514,24747.0,8
8,670,23041.0,9
9,2681,22850.0,10


### 2f. Products with average transaction above KES 1,300 (HAVING)

In [9]:
run('f')

Business question (f): Which products have an average transaction value
above KES 1,300? 



,product,avg_txn_value_kes,txn_count
0,Send Money,1355.3,4161
1,Bill Pay,1350.2,2388
2,Merchant Pay,1345.7,1789


### 2g. Users per value tier (CTE + CASE WHEN + LAG)

In [10]:
run('g')

Business question (g): How many users fall into each value tier
(High >= KES 10,000; Medium >= KES 4,000; otherwise Low), based on lifetime
transaction value? Users with no transactions count as Low (value 0).
LAG shows the gap in user count versus the tier listed just above. 



,value_tier,users,change_vs_tier_above
0,High,345,NaN
1,Medium,1317,972.0
2,Low,1338,21.0


### 2h. Month-over-month change in active users (CTE + LAG + CASE WHEN)

In [11]:
run('h')

Business question (h): How does the number of active users (distinct users
transacting) change month over month, in absolute and percentage terms?
NOTE: January (data starts 15 Jan) and June (data ends 13 Jun) are partial. 



,month,active_users,prev_active_users,change,pct_change,direction
0,2026-01,1055,NaN,NaN,NaN,n/a (first month)
1,2026-02,1596,1055.0,541.0,51.3,Growth
2,2026-03,1649,1596.0,53.0,3.3,Growth
3,2026-04,1704,1649.0,55.0,3.3,Growth
4,2026-05,1690,1704.0,-14.0,-0.8,Decline
5,2026-06,907,1690.0,-783.0,-46.3,Decline


**Reading 2h.** January starts on 15 Jan and June ends on 13 Jun, so those two months are partial. The +51% (Feb vs Jan) and -46% (Jun vs May) are artefacts of partial coverage, not real growth or collapse. The comparable full months (Feb to May) show active users rising about 3% a month and then flattening (-0.8% in May).

## 3. KPI scorecard

### Metric dictionary

| Metric | Definition | Calculation |
|---|---|---|
| Total value | Sum of KES moved through all transactions in the period (GMV, not revenue). | `SUM(amount)` |
| Number of transactions | Count of transaction records. | `COUNT(txn_id)` |
| Active users | Distinct users with at least one transaction in the period. | `COUNT(DISTINCT user_id)` |
| Average transaction value (ATV) | Mean KES per transaction. | `total value / number of transactions` |
| Avg transactions per active user | Transaction intensity among users who were active. | `number of transactions / active users` |
| MoM growth | Percentage change of a metric versus the previous calendar month. | `(this month - previous month) / previous month` |

Periods use calendar months of `txn_date`. Jan (starts 15th) and Jun (ends 13th) are partial, so MoM figures involving them are flagged.

In [12]:
u = pd.read_sql("SELECT * FROM users", con, parse_dates=["signup_date"])
t = pd.read_sql("SELECT * FROM transactions", con, parse_dates=["txn_date"])
t = t.merge(u, on="user_id")
t["month"] = t.txn_date.dt.to_period("M").astype(str)
t["pre_signup"] = t.txn_date < t.signup_date

def kpis(df):
    return pd.Series({
        "total_value_kes": df.amount.sum(),
        "transactions": len(df),
        "active_users": df.user_id.nunique(),
        "avg_txn_value_kes": df.amount.mean(),
        "txns_per_active_user": len(df) / df.user_id.nunique(),
    })

headline = pd.DataFrame({"all_transactions": kpis(t), "excl_pre_signup": kpis(t[~t.pre_signup])}).round(2)
print("Headline KPIs (whole period)"); display(headline)

monthly = t.groupby("month").apply(kpis, include_groups=False)
monthly["value_mom_pct"] = (monthly.total_value_kes.pct_change() * 100).round(1)
monthly["active_users_mom_pct"] = (monthly.active_users.pct_change() * 100).round(1)
monthly["txns_mom_pct"] = (monthly.transactions.pct_change() * 100).round(1)
monthly["coverage"] = ["partial (from 15 Jan)", "full", "full", "full", "full", "partial (to 13 Jun)"]
monthly.round(2)

Headline KPIs (whole period)


,all_transactions,excl_pre_signup
total_value_kes,15979309.00,10956778.00
transactions,12000.00,8263.00
active_users,2943.00,2716.00
avg_txn_value_kes,1331.61,1326.00
txns_per_active_user,4.08,3.04


,total_value_kes,transactions,active_users,avg_txn_value_kes,txns_per_active_user,value_mom_pct,active_users_mom_pct,txns_mom_pct,coverage
month,,,,,,,,,
2026-01,1733121.0,1308.0,1055.0,1325.02,1.24,NaN,NaN,NaN,partial (from 15 Jan)
2026-02,2993105.0,2272.0,1596.0,1317.39,1.42,72.7,51.3,73.7,full
2026-03,3199125.0,2394.0,1649.0,1336.31,1.45,6.9,3.3,5.4,full
2026-04,3262447.0,2470.0,1704.0,1320.83,1.45,2.0,3.3,3.2,full
2026-05,3365002.0,2475.0,1690.0,1359.60,1.46,3.1,-0.8,0.2,full
2026-06,1426509.0,1081.0,907.0,1319.62,1.19,-57.6,-46.3,-56.3,partial (to 13 Jun)


**Reading the scorecard.** Across the period PesaPap moved about KES 16.0M in 12,000 transactions, with 2,943 users active at least once, an average transaction of about KES 1,332 and about 4.1 transactions per active user. Among full-month comparisons (Mar, Apr, May) value grew 6.9%, 2.0% and 3.1% month on month, positive but slowing, while active users stalled in May (-0.8%). Feb vs Jan (+72.7%) and Jun vs May (-57.6%) are artefacts of partial months. Excluding pre-signup rows lowers value and transactions by about a third, so those figures are sensitive to the data issue above.

## 4. RFM segmentation

Uses post-signup transactions only. Reference date is the day after the last transaction (2026-06-14). Recency = days since last transaction (lower is better), Frequency = transaction count, Monetary = total KES. Each is scored 1 to 5 by quintile (rank-based, so ties are split deterministically).

In [13]:
clean = t[~t.pre_signup]
ref = t.txn_date.max() + pd.Timedelta(days=1)
rfm = clean.groupby("user_id").agg(last_txn=("txn_date", "max"),
                                   frequency=("txn_id", "count"),
                                   monetary=("amount", "sum"))
rfm["recency"] = (ref - rfm.last_txn).dt.days
rfm["R"] = pd.qcut(rfm.recency.rank(method="first"), 5, labels=[5, 4, 3, 2, 1]).astype(int)
rfm["F"] = pd.qcut(rfm.frequency.rank(method="first"), 5, labels=[1, 2, 3, 4, 5]).astype(int)
rfm["M"] = pd.qcut(rfm.monetary.rank(method="first"), 5, labels=[1, 2, 3, 4, 5]).astype(int)

R, F = rfm.R, rfm.F
rfm["segment"] = np.select(
    [(R >= 4) & (F >= 4), (R >= 3) & (F >= 3), (R >= 4) & (F <= 2), (R <= 2) & (F >= 3), (R <= 2) & (F <= 2)],
    ["Champions", "Loyal", "Promising", "At Risk", "Hibernating"], default="Needs Attention")

print(f"Users with at least one post-signup transaction: {len(rfm)} of {len(u)}")
seg = rfm.groupby("segment").agg(users=("monetary", "size"),
                                 total_value=("monetary", "sum"),
                                 avg_value=("monetary", "mean"),
                                 avg_recency_days=("recency", "mean"),
                                 avg_frequency=("frequency", "mean")).round(1)
seg["pct_of_users"] = (100 * seg.users / seg.users.sum()).round(1)
seg["pct_of_value"] = (100 * seg.total_value / seg.total_value.sum()).round(1)
seg.sort_values("total_value", ascending=False)

Users with at least one post-signup transaction: 2716 of 3000


,users,total_value,avg_value,avg_recency_days,avg_frequency,pct_of_users,pct_of_value
segment,,,,,,,
Champions,544,3504037.0,6441.2,7.7,4.9,20.0,32.0
Loyal,582,2784647.0,4784.6,16.8,3.5,21.4,25.4
At Risk,503,2497350.0,4964.9,49.6,3.8,18.5,22.8
Hibernating,583,1104381.0,1894.3,58.4,1.4,21.5,10.1
Promising,313,657759.0,2101.5,8.2,1.6,11.5,6.0
Needs Attention,191,408604.0,2139.3,22.9,1.6,7.0,3.7


**Segment rules.** Champions: R>=4 and F>=4. Loyal: R>=3 and F>=3 (not Champions). Promising: R>=4 and F<=2 (recent, few transactions). At Risk: R<=2 and F>=3 (used to transact often, now lapsing). Hibernating: R<=2 and F<=2. Needs Attention: everything else.

**Target first: At Risk.** These ~500 users transacted about 3.8 times each and about KES 5,000 on average (about 23% of all value), but have been silent for about 50 days. They are the best win-back opportunity: proven habit, high value, and still recent enough to recover, whereas Hibernating users are lower-value and colder. Champions need protecting but are already engaged, so a campaign there has the smallest incremental effect.

## 5. Monthly cohort retention

Cohort = month of signup (Jan to Apr 2026). Activity = at least one post-signup transaction in a calendar month. Retention(k) = active users in month k after signup month / cohort size.

**Incomplete cells are flagged with `*`:** (i) any cell in June 2026, because data ends on 13 Jun; (ii) month 0 of the January cohort, because transactions only start on 15 Jan. Cells beyond the data window are shown as blank (not yet observable). In addition, month 0 is partial for every cohort because users sign up throughout the month.

In [14]:
c = clean.copy()
c["cohort"] = c.signup_date.dt.to_period("M")
c["act_month"] = c.txn_date.dt.to_period("M")
c["offset"] = (c.act_month - c.cohort).apply(lambda x: x.n)
cohort_size = u.assign(cohort=u.signup_date.dt.to_period("M")).groupby("cohort").size()
active = c.groupby(["cohort", "offset"]).user_id.nunique().unstack()
ret = active.div(cohort_size, axis=0)

last_month = pd.Period("2026-06")
partial = pd.DataFrame(False, index=ret.index, columns=ret.columns)
for co in ret.index:
    for k in ret.columns:
        cal = co + k
        if cal > last_month:
            ret.loc[co, k] = np.nan
        elif cal == last_month or (co == pd.Period("2026-01") and k == 0):
            partial.loc[co, k] = True

shown = ret.apply(lambda col: col.map(lambda v: "" if pd.isna(v) else f"{100*v:.1f}%"))
shown = shown.where(~partial, shown + "*")
shown.insert(0, "cohort_size", cohort_size)
print("Cohort retention (% of cohort active). * = incomplete cell, blank = not yet observable")
shown

Cohort retention (% of cohort active). * = incomplete cell, blank = not yet observable


offset,cohort_size,0,1,2,3,4,5
cohort,,,,,,,
2026-01,773,24.7%*,53.7%,55.9%,58.3%,56.0%,33.5%*
2026-02,703,29.9%,54.9%,54.6%,59.0%,28.7%*,
2026-03,745,32.5%,55.4%,56.9%,30.2%*,,
2026-04,779,34.9%,53.7%,28.4%*,,,


In [15]:
fig_data = (100 * ret).round(1)
fig_data.style.background_gradient(cmap="Blues", axis=None).format("{:.1f}", na_rep="")

offset,0,1,2,3,4,5
cohort,,,,,,
2026-01,24.7,53.7,55.9,58.3,56.0,33.5
2026-02,29.9,54.9,54.6,59.0,28.7,
2026-03,32.5,55.4,56.9,30.2,,
2026-04,34.9,53.7,28.4,,,


**Reading the table.** In complete cells, about 54% to 59% of a cohort is active in any month after signup month, and it does not decay with age or differ by cohort. Month 0 is lower (25% to 35%) simply because users only have part of the month to transact, and the starred June cells (28% to 34%) are low because June is only 13 days long: neither is a real drop. The flat curve is expected here because transaction dates are generated independently of signup dates; real data would normally show retention falling with age. Starred cells should not be used for conclusions.

## 6. Experiment: simplified sign-up screen (group B) vs control (group A)

**Hypotheses.** H0: the conversion rate is the same in A and B (p_B - p_A = 0). H1: the conversion rates differ (p_B - p_A != 0), two-sided test at alpha = 0.05.

In [16]:
e = pd.read_sql('SELECT * FROM experiment', con)
g = e.groupby("group").converted.agg(n="count", conversions="sum", rate="mean")
print("Group sizes and conversion"); display(g)

# Balance check 1: sample-ratio test against an intended 50/50 split
srm = stats.chisquare(g.n)
print(f"Sample-ratio check (50/50): chi2 = {srm.statistic:.2f}, p = {srm.pvalue:.3f}")

# Balance check 2: covariates by group
eu = e.merge(u, on="user_id"); eu["signup_month"] = eu.signup_date.dt.to_period("M").astype(str)
bal = []
for col in ["county", "acquisition_channel", "signup_month"]:
    chi, p, _, _ = stats.chi2_contingency(pd.crosstab(eu[col], eu["group"]))
    bal.append({"covariate": col, "chi2": round(chi, 2), "p_value": round(p, 3)})
print("Covariate balance (small chi2 / large p = balanced)"); display(pd.DataFrame(bal))

Group sizes and conversion


,n,conversions,rate
group,,,
A,1443,151,0.104643
B,1557,215,0.138086


Sample-ratio check (50/50): chi2 = 4.33, p = 0.037
Covariate balance (small chi2 / large p = balanced)


,covariate,chi2,p_value
0,county,2.20,0.699
1,acquisition_channel,1.98,0.576
2,signup_month,0.33,0.953


**Balance check.** Group sizes are 1,443 (A) vs 1,557 (B). The covariate checks (county, channel, signup month) show no meaningful imbalance, so the groups look comparable on who they contain. The size split is slightly uneven: the 50/50 sample-ratio test gives p of about 0.04, i.e. the 114-user gap is a little larger than chance usually produces. With simple random assignment this happens about 1 time in 27, so I treat it as a flag rather than a failure, but the engineering team should confirm assignment was truly random. Conversion rates are rates per group, so unequal sizes do not bias them.

In [17]:
a, b = g.loc["A"], g.loc["B"]
pA, pB = a.rate, b.rate
abs_diff = pB - pA
rel_lift = abs_diff / pA

table = np.array([[a.conversions, a.n - a.conversions], [b.conversions, b.n - b.conversions]])
chi2, p_val, dof, _ = stats.chi2_contingency(table, correction=False)
chi2_y, p_yates, _, _ = stats.chi2_contingency(table, correction=True)

se = np.sqrt(pA * (1 - pA) / a.n + pB * (1 - pB) / b.n)
ci_lo, ci_hi = abs_diff - 1.96 * se, abs_diff + 1.96 * se

res = pd.DataFrame({
    "A (control)": [int(a.n), int(a.conversions), f"{100*pA:.2f}%"],
    "B (simplified)": [int(b.n), int(b.conversions), f"{100*pB:.2f}%"]},
    index=["users", "conversions", "conversion rate"])
display(res)
print(f"Absolute difference (B - A): {100*abs_diff:.2f} percentage points")
print(f"Relative lift: {100*rel_lift:.1f}%")
print(f"Chi-square (no continuity correction): chi2 = {chi2:.2f}, df = {dof}, p = {p_val:.4f}")
print(f"Chi-square (Yates correction):         chi2 = {chi2_y:.2f}, p = {p_yates:.4f}")
print(f"95% CI for difference: [{100*ci_lo:.2f}, {100*ci_hi:.2f}] percentage points")

,A (control),B (simplified)
users,1443,1557
conversions,151,215
conversion rate,10.46%,13.81%


Absolute difference (B - A): 3.34 percentage points
Relative lift: 32.0%
Chi-square (no continuity correction): chi2 = 7.82, df = 1, p = 0.0052
Chi-square (Yates correction):         chi2 = 7.51, p = 0.0061
95% CI for difference: [1.01, 5.67] percentage points


**Interpretation.** Group B converted at 13.8% versus 10.5% for group A: a 3.3 percentage-point gain, or a 32% relative lift. The chi-square test gives p of about 0.005 (0.006 with Yates correction), well below 0.05, so we reject H0: a gap this large would be very unlikely if the two screens performed the same. The 95% confidence interval for the difference runs from about +1.0 to +5.7 percentage points. It excludes zero, consistent with the test, but it is wide: the true gain could be as small as 1 point (about 10% relative) or as large as almost 6 points. Statistical significance does not tell us the gain is large, only that it is very unlikely to be zero.

## 7. Business impact of rolling out variant B

**Assumptions (change them to test sensitivity).**
1. Sign-up volume continues at the observed rate: 3,000 sign-ups over 120 days, about 760 per month.
2. The experiment effect (conversion rate A to B) persists after rollout, at the point estimate and at the CI bounds.
3. "Converted" means the user becomes an activated, transacting customer. Value per converted user = average post-signup transaction value per transacting user in this data.
4. PesaPap earns a take rate of 1.5% of transaction value (an **assumption**; replace with the real figure). Revenue = value x take rate.
5. Incremental converters behave like average converters (likely optimistic, as marginal users are often less engaged).

In [18]:
monthly_signups = len(u) / 120 * 30.4
value_per_user = clean.groupby("user_id").amount.sum().mean()   # KES over the observed window
take_rate = 0.015

rows = []
for label, d in [("Low (CI lower)", ci_lo), ("Point estimate", abs_diff), ("High (CI upper)", ci_hi)]:
    extra_users_month = monthly_signups * d
    extra_users_year = extra_users_month * 12
    gmv_year = extra_users_year * value_per_user
    rows.append({"scenario": label, "uplift_pp": round(100 * d, 2),
                 "extra_converters_per_month": round(extra_users_month, 1),
                 "extra_converters_per_year": round(extra_users_year),
                 "extra_GMV_per_year_KES": round(gmv_year),
                 "extra_revenue_per_year_KES": round(gmv_year * take_rate)})
print(f"Monthly sign-ups: {monthly_signups:.0f} | value per converted user: KES {value_per_user:,.0f} | take rate: {take_rate:.1%}")
impact = pd.DataFrame(rows); display(impact)

print("Sensitivity: point-estimate extra GMV per year if sign-ups scale (KES)")
for mult in [1, 2, 5]:
    print(f"  {mult}x sign-ups ({monthly_signups*mult:,.0f}/month): KES {monthly_signups*mult*abs_diff*12*value_per_user:,.0f}")

Monthly sign-ups: 760 | value per converted user: KES 4,034 | take rate: 1.5%


,scenario,uplift_pp,extra_converters_per_month,extra_converters_per_year,extra_GMV_per_year_KES,extra_revenue_per_year_KES
0,Low (CI lower),1.01,7.7,92,373019,5595
1,Point estimate,3.34,25.4,305,1230418,18456
2,High (CI upper),5.67,43.1,518,2087816,31317


Sensitivity: point-estimate extra GMV per year if sign-ups scale (KES)
  1x sign-ups (760/month): KES 1,230,418
  2x sign-ups (1,520/month): KES 2,460,836
  5x sign-ups (3,800/month): KES 6,152,089


**Reading the impact.** At today's volume the simplified screen adds roughly 25 converted users a month (about 305 a year, range about 90 to 520), worth roughly KES 1.2M of annual transaction value (range about 0.4M to 2.1M) and roughly KES 18,000 of revenue at an assumed 1.5% take rate. The effect is statistically solid but small in absolute shillings at this scale; it becomes material only if sign-up volume grows or the new screen is cheap to ship. Because the change is a UI simplification with low cost and low downside, the case for rolling out rests mainly on cost being near zero.